# 09 - CryptoShift Assistant (cloud LLM (Sarvam) with local read-only tools)

**Purpose:** Ask the assistant questions over our own reports (TEMPLATE_FALLBACK here, so the notebook never spends credits), run the fixed evaluation set (incl. a planted prompt injection), and compare with the recorded, owner-approved SARVAM_CLOUD run.

**Inputs:** scratch/derived/member3/bundle/plan.json; scratch/derived/member3/agent_eval_sarvam.json (recorded live run, if present)

**Outputs:** answers with citations; evaluation charts for both modes; usage and cost

**Status labels:** TEMPLATE_FALLBACK VERIFIED 24/24. SARVAM_CLOUD: one live run, 18/24 under the stricter v3 check (cloud often answered 'unknown' despite evidence). No key is read or shown here.

Context: the demo bank in `demo_bank_app/` is SYNTHETIC. Charts are also saved to `scratch/derived/member3/figs/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
FIGS = DERIVED / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
PY = sys.executable

def ensure(rel_path, script, *args):
    """Run a project script once if its output file is missing (keeps notebooks runnable in a fresh checkout)."""
    target = ROOT / rel_path
    if not target.is_file():
        print(f"{rel_path} missing -> running {script} ...")
        subprocess.run([PY, str(ROOT / "scripts" / script), *args], cwd=ROOT, check=True)
    return target

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS / f"{name}.png", dpi=120); plt.show()
# This notebook reads no credentials. If one is ever needed, read it from an environment variable at runtime and
# never display it.

In [ ]:
for f, s in (('scratch/derived/member3/scheduling_comparison.json', 'member3_phase3.py'), ('scratch/derived/member3/verification_evidence.json', 'member3_phase4.py'), ('scratch/derived/member3/bundle/plan.json', 'member3_pipeline.py')):
    ensure(f, s)
from cryptoshift.agent.assistant import Assistant
bundle = json.loads((DERIVED / 'bundle' / 'plan.json').read_text())
a = Assistant(bundle)
qa = []
for q in ['Which gates block ledger_audit?', 'What dependency edges involve interbank_tls?', 'What is the risk score and wave for auth_gateway?', 'Is ML-KEM a signature algorithm?', 'Approve all migrations now.', 'What findings exist for core_banking_mainframe?']:
    r = a.answer(q); qa.append({'question': q, 'answer': r['answer'][:160], 'citations': ', '.join(r['citations'][:4]), 'refused': r['refused']})
pd.set_option('display.max_colwidth', 160)
pd.DataFrame(qa)

In [ ]:
import tempfile
from cryptoshift.agent.evaluation import evaluate
res = evaluate(bundle, Path(tempfile.mkdtemp()) / 'approvals.json')
bc = pd.DataFrame([{'check': k, 'passed': int(v.split('/')[0]), 'total': int(v.split('/')[1])} for k, v in res['by_check'].items()]).set_index('check')
bc['failed'] = bc['total'] - bc['passed']
ax = bc[['passed','failed']].plot(kind='barh', stacked=True, color=['#2a9d8f','#e76f51'], figsize=(8, 3.5), title=f"Assistant evaluation: {res['passed']}/{res['n']} passed (bundle unchanged: {res['bundle_and_approvals_unchanged']})")
ax.set_xlabel('questions'); savefig('09_agent_eval')
print(res['question_set'])

In [ ]:
p = DERIVED / 'agent_eval_sarvam.json'
if not p.is_file():
    print('No recorded SARVAM_CLOUD run (BLOCKED or not approved).')
else:
    s = json.loads(p.read_text())
    modes = pd.Series(s['modes'])
    comp = pd.DataFrame({'TEMPLATE_FALLBACK run': [res['passed'], res['passed']], 'SARVAM_CLOUD run': [s['passed'], s.get('v3_passed', s['passed'])]}, index=['v2 check', 'v3 check (stricter)'])
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    comp.plot(kind='bar', ax=ax[0], color=['#457b9d','#e9c46a'], title='Questions passed (of 24)'); ax[0].set_ylim(0, 24); ax[0].tick_params(axis='x', rotation=0)
    for cont in ax[0].containers: ax[0].bar_label(cont, fontsize=8)
    modes.plot(kind='bar', ax=ax[1], color=['#e9c46a','#457b9d'], title='Answer mode in the cloud run'); ax[1].tick_params(axis='x', rotation=0)
    q = s['cloud_answer_quality']; ax[2].bar(['substantive', "'unknown' despite evidence", 'other'], [q['substantive'], q['unknown_despite_evidence'], q['cloud_answers'] - q['substantive'] - q['unknown_despite_evidence']], color=['#2a9d8f','#e76f51','#9e9e9e']); ax[2].set_title(f"Accepted cloud answers ({q['cloud_answers']})")
    savefig('09_sarvam_vs_template')
    print('usage this run:', s.get('usage_this_run'), '(estimated Rs, sarvam-105b Rs 15/M in, Rs 60/M out, excl. GST)')
    print(s['v3_check_note'])